# 05a - Stacked Area (Fractional Budget Over Time)
Run `05-00-shared-data.ipynb` first to load data.

In [ ]:
%run 05-00-shared-data.ipynb
from utilities.style_profiles import apply_publication_style, save_fig, SINGLE_COL_IN, FULL_COL_IN, MAX_H_IN, MM, PROC_COLORS, proc_color, ENERGY_PROCESSES, HEAT_RELEASE_PROCESSES, HEAT_CONSUME_PROCESSES
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import numpy as np
apply_publication_style()

---
## View A – Normalised stacked area: fractional budget over time

For a given station (height-averaged), split tendencies into **sources**
(positive rates) and **sinks** (negative rates). Compute each process's
fractional contribution at every timestep. Time is coarsened to reduce
noise.

**What this view shows:** Height-averaged, bin-summed process tendencies, split into **sources** (positive) and **sinks** (negative), normalised so that at each time the stacked area shows each process's **fraction** of the total source or sink. Time is coarsened to reduce noise. The dashed line is seeding start (12:30).

**Answering "which processes contribute most?"** The stacked fractions are exactly the process tendency terms (e.g. CONDNFROD, IMMERN, DEPON, …) normalised per timestep. So the **relative area** of each colour is that process's contribution. For **ice number**: the dark-blue **CONDENSATION** slice is **CONDNFROD** — in the source code this is **redistribution only** (sum over bins = 0), so it does **not** add total ice number. For total ice number growth, treat it as zero and look at the other processes (e.g. immersion freezing, contact freezing, deposition nucleation). For **ice mass**, the condensation term is depositional growth + bin shift combined and does contribute to mass growth.

In [ ]:
# View A — Stacked area (sources/sinks) and rate-only panels. Data stays xarray until plot (.values).
from collections import namedtuple

def _minutes_since_seed(time_vals):
    """Convert datetime array to minutes since seed_start (12:30 = 0)."""
    t = np.asarray(time_vals).astype("datetime64[us]")
    t0 = np.datetime64(seed_start).astype("datetime64[us]")
    return (t - t0) / np.timedelta64(1, "m")

# View A energy cues: hatch + edge for latent-heat release vs consume (from style_profiles sets).
HATCH_HEAT_RELEASE, HATCH_HEAT_CONSUME = "///", "///"
EDGE_HEAT_RELEASE = (0.85, 0.2, 0.2, 0.5)
EDGE_HEAT_CONSUME = (0.2, 0.4, 0.9, 0.5)
ENERGY_HATCH_LW_RELEASE, ENERGY_HATCH_LW_CONSUME = 0.8, 0.5

def _apply_energy_hatch(patch, proc):
    """Apply heat-release (red) or heat-consume (blue) hatch/edge to a polygon patch."""
    if proc in HEAT_RELEASE_PROCESSES:
        patch.set_hatch(HATCH_HEAT_RELEASE)
        patch.set_edgecolor(EDGE_HEAT_RELEASE)
        patch.set_linewidth(ENERGY_HATCH_LW_RELEASE)
    elif proc in HEAT_CONSUME_PROCESSES:
        patch.set_hatch(HATCH_HEAT_CONSUME)
        patch.set_edgecolor(EDGE_HEAT_CONSUME)
        patch.set_linewidth(ENERGY_HATCH_LW_CONSUME)

def _prep_fractions(rates_dict, si):
    """Height-avg, time-coarsened source/sink fractions. Returns named tuple (time, procs, frac_pos, frac_neg, tot_pos, tot_neg)."""
    procs = list(rates_dict.keys())
    combo = xr.Dataset({
        p: rates_dict[p].isel(station=si).mean("height_level")
           .resample(time=time_coarsen).mean().compute()
        for p in procs})
    pos = combo.clip(min=0)
    neg = (-combo).clip(min=0)
    tot_pos = pos.to_array("proc").sum("proc")
    tot_neg = neg.to_array("proc").sum("proc")
    frac_pos = (pos / tot_pos.where(tot_pos > 0)).fillna(0)
    frac_neg = (neg / tot_neg.where(tot_neg > 0)).fillna(0)
    PrepResult = namedtuple("PrepResult", "time procs frac_pos frac_neg tot_pos tot_neg")
    return PrepResult(
        combo.time.values, procs, frac_pos, frac_neg, tot_pos.values, tot_neg.values
    )

def _format_time_axis(ax):
    """Format x-axis as minutes since seeding: 0 and ±N."""
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: "0" if x == 0 else f"{int(round(x)):+d}"))

def _draw_seed_vline_and_floor_band(ax, time_min, tot_raw, y_lo, y_hi):
    """Draw seeding start vline and grey band where |Σ rate| < rate_floor."""
    ax.axvline(0, color="k", lw=1.2, ls="--", label="seeding start")
    ax.fill_between(time_min, y_lo, y_hi, where=tot_raw < rate_floor,
                     color="0.85", alpha=0.3, zorder=5, label=f"|Σrate| < {rate_floor:.0e}")

def _draw_log_rate_curve(ax, time_min, tot_raw, unit_label, is_sink, show_ylabel, as_twin=False):
    """Plot log10(|Σ rate|) or -log10; optionally on a twin y-axis. Returns ax used for the curve."""
    log_tot = np.log10(np.where(tot_raw > 0, tot_raw, np.nan))
    y = -log_tot if is_sink else log_tot
    ref = -np.log10(rate_floor) if is_sink else np.log10(rate_floor)
    if as_twin:
        ax2 = ax.twinx()
        ax2.plot(time_min, y, color="k", lw=0.8, alpha=0.85, zorder=6)
        ax2.axhline(ref, color="k", lw=0.6, ls=":", alpha=0.5)
        if show_ylabel:
            sym = r"-\log_{10}" if is_sink else r"\log_{10}"
            ax2.set_ylabel(rf"${sym} |Σ\mathrm{{rate}}|$ / ({unit_label})")
        return ax2
    ax.plot(time_min, y, color="k", lw=0.8, alpha=0.9, zorder=6)
    ax.axhline(ref, color="k", lw=0.6, ls=":", alpha=0.5)
    if show_ylabel:
        sym = r"-\log_{10}" if is_sink else r"\log_{10}"
        ax.set_ylabel(rf"${sym} |Σ\mathrm{{rate}}|$ / ({unit_label})")
    valid = np.isfinite(y)
    if valid.any():
        ymin, ymax = np.nanmin(y[valid]), np.nanmax(y[valid])
        margin = (ymax - ymin) * 0.05 or 0.5
        ax.set_ylim(ymin - margin, ymax + margin)
    return ax

def _fill_panel_stacked(ax, frac_ds, procs, tot_raw, time_min, unit_label, is_sink,
                        title="", show_twin_ylabel=True, draw_twin=True):
    """Single stacked-area panel for sources (0–1) or sinks (-1–0). Optionally twin with log |rate|."""
    y_lo, y_hi = (-1, 0) if is_sink else (0, 1)
    y_base = np.zeros_like(time_min, dtype=float)
    for p in procs:
        seg = frac_ds[p].values  # positive fraction in both cases (frac_pos or frac_neg)
        y_top = y_base + seg
        if is_sink:
            y_a, y_b = -y_top, -y_base   # map [0,1] running sum to [-1,0] for display
        else:
            y_a, y_b = y_base, y_top
        coll = ax.fill_between(time_min, y_a, y_b, color=proc_color(p), alpha=0.85, label=p)
        _apply_energy_hatch(coll, p)
        y_base = y_top
    _draw_seed_vline_and_floor_band(ax, time_min, tot_raw, y_lo, y_hi)
    ax.set(ylim=(y_lo, y_hi), ylabel="Fraction", title=title)
    _format_time_axis(ax)
    if draw_twin:
        _draw_log_rate_curve(ax, time_min, tot_raw, unit_label, is_sink, show_twin_ylabel, as_twin=True)

def _fill_panel_rate(ax, tot_raw, time_min, unit_label, is_sink, show_ylabel=True):
    """Rate-only panel: log(|Σ rate|) curve, no stacked area (mode='rate')."""
    ax.axvline(0, color="k", lw=1.2, ls="--", label="seeding start")
    _format_time_axis(ax)
    _draw_log_rate_curve(ax, time_min, tot_raw, unit_label, is_sink, show_ylabel, as_twin=False)

def plot_stacked_area_liquid_ice(rates_N_liq, rates_Q_liq, rates_N_ice, rates_Q_ice, si=stn_idx, mode="fraction", range_key=None):
    """Single figure: row1 = sources, row2 = sinks. mode='fraction' (stacked area) or 'rate' (log |Σ rate| only)."""
    
    import matplotlib.patches as mpatches
    prep_liq_N = _prep_fractions(rates_N_liq, si)
    prep_liq_Q = _prep_fractions(rates_Q_liq, si)
    prep_ice_N = _prep_fractions(rates_N_ice, si)
    prep_ice_Q = _prep_fractions(rates_Q_ice, si)
    preps = [prep_liq_N, prep_liq_Q, prep_ice_N, prep_ice_Q]
    units = [unit_N, unit_Q, unit_N, unit_Q]
    all_procs = set()
    for prep in preps:
        all_procs |= set(prep.procs)
    procs_ordered = [p for p in PROC_COLORS if p in all_procs]
    if mode == "fraction":
        frac_panels = [(prep.frac_pos, prep.procs) for prep in preps] + [(prep.frac_neg, prep.procs) for prep in preps]
        def _max_frac(p):
            return max((float(ds[p].max().values) if p in procs else 0.0) for ds, procs in frac_panels)
        procs_ordered = [p for p in procs_ordered if _max_frac(p) > 0.001]

    time_min = _minutes_since_seed(prep_liq_N.time)
    is_fraction = mode == "fraction"
    gridspec_kw = {"hspace": 0, "wspace": 0.05 if is_fraction else 0.15}
    fig, axes = plt.subplots(
        2, 4,
        figsize=(FULL_COL_IN, min(1.75 * 52 * MM, MAX_H_IN)),
        sharex="row", sharey="row" if is_fraction else False,
        constrained_layout=True,
        gridspec_kw=gridspec_kw,
    )
    if is_fraction:
        for j, (prep, ulbl) in enumerate(zip(preps, units)):
            _fill_panel_stacked(
                axes[0, j], prep.frac_pos, prep.procs, prep.tot_pos, time_min, ulbl,
                is_sink=False, show_twin_ylabel=False, draw_twin=False,
            )
            axes[0, j].set_title("")
        for j, (prep, ulbl) in enumerate(zip(preps, units)):
            _fill_panel_stacked(
                axes[1, j], prep.frac_neg, prep.procs, prep.tot_neg, time_min, ulbl,
                is_sink=True, show_twin_ylabel=False, draw_twin=False,
            )
            axes[1, j].set_title("")
    else:
        for j, (prep, ulbl) in enumerate(zip(preps, units)):
            _fill_panel_rate(axes[0, j], prep.tot_pos, time_min, ulbl, is_sink=False, show_ylabel=(j == 3))
            axes[0, j].set_title("")
        for j, (prep, ulbl) in enumerate(zip(preps, units)):
            _fill_panel_rate(axes[1, j], prep.tot_neg, time_min, ulbl, is_sink=True, show_ylabel=(j == 3))
            axes[1, j].set_title("")

    # Row 0: no x ticks/labels
    for j in range(4):
        axes[0, j].set_xticklabels([])
        axes[0, j].set_xlabel("")
        phase = "Liquid" if j % 2 == 0 else "Ice"
        moment = "Number" if j // 2 == 0 else "Mass"
        axes[0, j].set_title(f"{phase} {moment}")
    axes[0, 0].set_ylabel("sources", x=-0.05)
    for i in range(2):
        for j in range(1, 4):
            axes[i, j].set_ylabel("")

    axes[1, 0].set_ylabel("sinks", x=0.05)

    if is_fraction:
        handles = [mpatches.Patch(color=proc_color(p), label=p.replace("_", " "), alpha=0.85) for p in procs_ordered]
        h_release = mpatches.Patch(facecolor="white", edgecolor=EDGE_HEAT_RELEASE, hatch=HATCH_HEAT_RELEASE,
                                   linewidth=ENERGY_HATCH_LW_RELEASE, label="Heat release (e.g. cond., dep., freezing)")
        h_consume = mpatches.Patch(facecolor="white", edgecolor=EDGE_HEAT_CONSUME, hatch=HATCH_HEAT_CONSUME,
                                   linewidth=ENERGY_HATCH_LW_CONSUME, label="Heat consume (e.g. melting)")
        h_riming_note = mpatches.Patch(facecolor="none", edgecolor="none", label="¹Riming only if sufficient liquid shell: |QF|/|QFW| < 0.7")
        handles = handles + [h_release, h_consume, h_riming_note]
        ncol_legend = (len(handles) + 1) // 2
        fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.015), frameon=False, ncol=ncol_legend)
    fig.supxlabel("Time since seeding start / (min)")
    if is_fraction:
        fig.text(-0.001, 0.5, "Process fraction", rotation=90, ha="center", va="center", transform=fig.transFigure)
    else:
        fig.text(0.98, 0.5, r"$\log_{10}$ |Σ rate|", rotation=90, ha="center", va="center", transform=fig.transFigure)
    
    fig.suptitle(
        f"View A — Liquid & ice process fractions ({range_key or active_range_key})\n{stn_label(si, station_labels)}  ·  {exp_label}",
        fontweight="semibold",
    )
    plt.show()
    return fig, axes

idx_dbg = None
# if "rates_by_exp" in dir() and rates_by_exp:
#     for eid in plot_exp_ids[:idx_dbg]:
#         R = rates_by_exp[eid]
#         for range_key in plot_range_keys:
#             Rsel = select_rates_for_range(R, range_key)
#             exp_label = R["exp_label"]
#             for si in plot_stn_ids:
#                 fig, axes = plot_stacked_area_liquid_ice(
#                     Rsel["rates_N_liq"], Rsel["rates_Q_liq"], Rsel["rates_N_ice"], Rsel["rates_Q_ice"],
#                     si=si, range_key=range_key
#                 )
#                 save_fig(fig, f"stacked_area_liquid_ice_exp{eid}_stn{si}_{range_key}", "png", "./")


***View A — Fractional process budget over time.** Normalised stacked-area plots showing the temporal evolution of each microphysical process's fractional contribution to the total source (top panel) and total sink (bottom panel) budget. Rates are height-averaged and bin-summed over the respective spectral range (cloud bins 30–49 or precipitation bins 50–65), then time-coarsened to 1 min. A thin black line on the twin axis traces $\log_{10}$ of the total absolute rate to provide magnitude context; grey shading masks periods where the budget is physically negligible ($|Σ\,\mathrm{rate}| < 10^{-12}$). Seeding onset (12:30 UTC) is marked by a dashed vertical line. Six figures are produced — one for each combination of spectrum (liquid, ice, precipitation) and tendency kind (number, mass).*

**What the plot showed:** View A — Liquid & ice process fractions (cloud bins): four columns (Liquid Number, Ice Number, Liquid Mass, Ice Mass), each with sources (top, 0–1) and sinks (bottom, 0 to −1) over time since seeding.

**Answer (Ice Number):** The dark-blue **CONDENSATION** in the **Ice Number sources** panel is CONDNFROD. In COSMO-SPECS this is **bin reclassification only** (particles moving between size bins when they grow by deposition); **sum over all bins = 0**, so it does **not** increase total ice crystal number. The processes that actually add ice number are the other colours (e.g. immersion freezing, contact freezing, deposition nucleation). After seeding, **immersion freezing** (flare INP → ice) is the main nucleation source for extra ice number; see `docs/ice_number_budget_and_seeding.md`.

**Answer (Ice Mass):** The condensation slice for ice mass is depositional growth + shift combined; it is a real contribution to ice mass growth.

**Answer (Liquid):** For liquid number/mass, condensation is the usual vapor–liquid process; interpretation is standard.